# 19 — Does the blend help FD004 too?

**Why this notebook exists.** Notebook 18 found that averaging the Random Survival Forest with a gradient-boosting regressor improved FD001's point estimate (official test NASA 3.12 vs 4.05 per engine) and the retraining gate promoted it. FD004 was not part of that experiment, so it kept the RSF-only point estimate and the two deployed models differed for no tested reason. FD004 is a harder problem: six operating conditions, two fault modes, and a regime-normalization step FD001 doesn't need. The blend could help there, do nothing, or hurt; this notebook finds out with the same protocol, rather than copying FD001's result.

**Candidates, fixed before running:**

| Candidate | What it is |
|---|---|
| RSF (deployed) | the FD004 deployed model: leaf 5, `row_stride=15`, cap 125 (baseline) |
| Blend RSF + GB | the average of that RSF and `HistGradientBoostingRegressor` (300 iterations, learning rate 0.05, not tuned) on the same features and capped target, fit on the same `row_stride` rows: exactly what `export_deployment_artifacts --point-model blend` builds |
| Gradient boosting alone | reference only, as in notebook 18 |

The cap candidates of notebook 18 are not repeated: they were rejected on FD001 for a structural reason (they win only by predicting more life for very young engines), and this notebook asks one question.

**Protocol:** identical to notebook 18. 5 folds by engine; each held-out engine scored at 5 random points between 30% and 90% of its life. The regime clusters and normalization statistics are fit on all training engines, as in notebooks 10 and 15: that step is unsupervised (it never sees remaining life) and it is the same for both candidates, so it cannot favour either.

**Selection rule (notebook 18's, unchanged):** the blend replaces the RSF only if (a) its mean NASA score per engine is at least 5% better, (b) it wins in at least 4 of 5 folds, (c) it is not worse on held-out points with at most 150 cycles left, and (d) its RMSE is not more than 5% worse.

**Test-set rule, fixed in advance:** if selected, the blend is deployed only if its official-test NASA score per engine is lower than the RSF's *and* it then passes the retraining gate. If it is not selected, the test set is still scored once, for the record, but the result changes nothing.

**Late-error severity.** After notebook 18 the obvious question was: the blend was late more often on FD001 (62% vs 56% of engines), so is it less safe? The late rate counts a 1-cycle overshoot the same as a 30-cycle one, so this notebook also reports how *large* the late errors are, overall and near failure.

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sksurv.metrics import concordance_index_censored

sys.path.append(str(Path("..") / "src"))
from turbofan_rul import (cross_validation, data_loading, evaluation, export_deployment_artifacts as export,
                          features, inference, labeling, models, plotting)

plotting.style()
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "models"
SUBSET, CAP, N_DRAWS = "FD004", 125, 5
LEAF = export.RSF_MIN_SAMPLES_LEAF_BY_SUBSET[SUBSET]
STRIDE = export.ROW_STRIDE_BY_SUBSET[SUBSET]
RECOMPUTE = False   # True re-fits every fold model

cmapss = data_loading.load_subset(SUBSET, data_dir=DATA_DIR)
kept, train_norm, _, _ = export._prepare_regime_clustered(SUBSET, cmapss)
META = {"unit_number", "time_cycles", "RUL", "duration", "event"}
print(f"{cmapss.train.unit_number.nunique()} training engines, {len(kept)} sensors kept, leaf {LEAF}, stride {STRIDE}")

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"NASA / engine": evaluation.nasa_score_mean(y, p), "RMSE": evaluation.rmse(y, p),
            "MAE": evaluation.mae(y, p), "late rate": evaluation.late_prediction_rate(y, p),
            "C-index": concordance_index_censored(np.ones(len(y), bool), y, -p)[0]}

def late_severity(y, p, near=25):
    y, p = np.asarray(y, float), np.asarray(p, float)
    d = p - y
    nf = y <= near
    return {"late (any)": int((d > 0).sum()), "mean late error": d[d > 0].mean(),
            "late by >20": int((d > 20).sum()), "late by >30": int((d > 30).sum()), "worst late": d.max(),
            f"near failure (<= {near}): late by >10": int((d[nf] > 10).sum()),
            f"near failure (<= {near}): worst late": d[nf].max()}

249 training engines, 17 sensors kept, leaf 5, stride 15


## 1. Cross-validated candidates

In [2]:
oof_path = PROCESSED_DIR / "point_candidates_oof_fd004.csv"
if oof_path.exists() and not RECOMPUTE:
    oof = pd.read_csv(oof_path)
else:
    units = train_norm.unit_number.unique()
    featured_all = features.build_feature_matrix(train_norm, kept_sensors=kept)
    out = []
    for fold, (tr_pos, te_pos) in enumerate(GroupKFold(n_splits=5).split(units, groups=units)):
        t0 = time.time()
        tr_raw = train_norm[train_norm.unit_number.isin(units[tr_pos])]
        te_raw = train_norm[train_norm.unit_number.isin(units[te_pos])]
        te_feat = featured_all[featured_all.unit_number.isin(units[te_pos])]
        evals = pd.concat([cross_validation.build_truncated_eval_frame(
            te_feat, cross_validation.sample_truncation_points(te_raw, random_state=42 + k)).assign(draw=k)
            for k in range(N_DRAWS)], ignore_index=True)
        train = features.build_feature_matrix(labeling.build_train_survival_frame(tr_raw, cap=CAP), kept_sensors=kept)
        cols = [c for c in train.columns if c not in META]
        rsf = models.fit_random_survival_forest(train, cols, row_stride=STRIDE, min_samples_leaf=LEAF,
                                                duration_eps=export.DURATION_EPS, random_state=export.RANDOM_STATE)
        sub = train.iloc[::STRIDE]
        gbm = HistGradientBoostingRegressor(**export.GBM_PARAMS).fit(sub[cols], sub["duration"])
        out.append(pd.DataFrame({"fold": fold, "unit_number": evals.unit_number.values, "draw": evals.draw.values,
                                 "true_rul": evals.true_rul.values,
                                 "rsf": models.predict_rul_from_survival(rsf, evals[cols]).pred_rul.values,
                                 "gbm": np.clip(gbm.predict(evals[cols]), 0, None)}))
        print(f"fold {fold}: {len(evals)} held-out points, {time.time() - t0:.0f}s")
    oof = pd.concat(out, ignore_index=True)
    oof["blend"] = (oof.rsf + oof.gbm) / 2
    oof.to_csv(oof_path, index=False)

CANDIDATES = {"RSF (deployed)": "rsf", "Gradient boosting (reference)": "gbm", "Blend RSF + GB": "blend"}
print(len(oof), "held-out points;", oof.groupby("fold").size().to_dict())

fold 0: 250 held-out points, 139s


fold 1: 250 held-out points, 172s


fold 2: 250 held-out points, 198s


fold 3: 250 held-out points, 171s


fold 4: 245 held-out points, 176s
1245 held-out points; {0: 250, 1: 250, 2: 250, 3: 250, 4: 245}


In [3]:
def summarise(frame):
    rows = {}
    for name, col in CANDIDATES.items():
        per_fold = frame.groupby("fold").apply(lambda g: evaluation.nasa_score_mean(g.true_rul, g[col]), include_groups=False)
        near = frame[frame.true_rul <= 150]
        rows[name] = {**metrics(frame.true_rul, frame[col]),
                      "NASA per fold": per_fold.round(2).tolist(),
                      "NASA / engine, true RUL <= 150": evaluation.nasa_score_mean(near.true_rul, near[col]),
                      "RMSE, true RUL <= 150": evaluation.rmse(near.true_rul, near[col])}
    return pd.DataFrame(rows).T

cv = summarise(oof)
display(cv.drop(columns="NASA per fold").astype(float).round(3))
display(cv[["NASA per fold"]])
display(pd.DataFrame({n: late_severity(oof.true_rul, oof[c]) for n, c in CANDIDATES.items()}).T.astype(float).round(2))

,NASA / engine,RMSE,MAE,late rate,C-index,"NASA / engine, true RUL <= 150","RMSE, true RUL <= 150"
RSF (deployed),140772.541,37.170,26.024,0.523,0.816,14.858,22.234
Gradient boosting (reference),149831.347,35.474,23.653,0.508,0.836,8.408,19.302
Blend RSF + GB,144695.046,35.997,24.485,0.525,0.833,9.596,20.151


,NASA per fold
RSF (deployed),"[7004.44, 693346.73, 181.45, 201.77, 319.25]"
Gradient boosting (reference),"[21990.08, 723602.19, 156.44, 134.94, 282.11]"
Blend RSF + GB,"[12138.25, 707827.87, 165.77, 158.41, 296.97]"


,late (any),mean late error,late by >20,late by >30,worst late,near failure (<= 25): late by >10,near failure (<= 25): worst late
RSF (deployed),651.0,18.07,230.0,125.0,67.27,21.0,29.82
Gradient boosting (reference),633.0,15.03,182.0,80.0,67.42,9.0,23.35
Blend RSF + GB,654.0,15.93,202.0,87.0,67.35,18.0,21.29


In [4]:
base, r = cv.loc["RSF (deployed)"], cv.loc["Blend RSF + GB"]
checks = {
    "(a) NASA >=5% better": r["NASA / engine"] <= 0.95 * base["NASA / engine"],
    "(b) wins >=4 of 5 folds": int((np.array(r["NASA per fold"]) < np.array(base["NASA per fold"])).sum()) >= 4,
    "(c) not worse at true RUL <= 150": r["NASA / engine, true RUL <= 150"] <= base["NASA / engine, true RUL <= 150"],
    "(d) RMSE at most 5% worse": r["RMSE"] <= 1.05 * base["RMSE"],
}
display(pd.Series(checks, name="Blend RSF + GB").to_frame().T)
selected = all(checks.values())
print("Selected on cross-validation:", "Blend RSF + GB" if selected else "none -- the deployed RSF stays")

,(a) NASA >=5% better,(b) wins >=4 of 5 folds,(c) not worse at true RUL <= 150,(d) RMSE at most 5% worse
Blend RSF + GB,False,False,True,True


Selected on cross-validation: none -- the deployed RSF stays


## 2. The official test set (once)

In [5]:
bundle = inference.load_deployment_bundle(MODELS_DIR / "fd004")
true_rul = cmapss.rul.set_index("unit_number")["RUL"]

# Gradient boosting fit on all training engines exactly as the export would.
train_full = features.build_feature_matrix(labeling.build_train_survival_frame(train_norm, cap=CAP), kept_sensors=kept)
sub = train_full.iloc[::STRIDE]
gbm_full = HistGradientBoostingRegressor(**export.GBM_PARAMS).fit(sub[bundle.full_feature_cols], sub["duration"])

rows = []
test_feat = inference.featurize(bundle, cmapss.test).groupby("unit_number").tail(1).set_index("unit_number")
for unit, engine in cmapss.test.groupby("unit_number"):
    r = inference.predict(bundle, engine.sort_values("time_cycles")[["time_cycles"] + bundle.raw_passthrough_cols])
    rows.append({"unit": unit, "true": float(true_rul.loc[unit]),
                 "rsf": r["point_estimate_components"]["random_survival_forest"] or r["predicted_rul_cycles"]})
test_pred = pd.DataFrame(rows)
test_pred["gbm"] = np.clip(gbm_full.predict(test_feat.loc[test_pred.unit, bundle.full_feature_cols]), 0, None)
test_pred["blend"] = (test_pred.rsf + test_pred.gbm) / 2

confirm = pd.DataFrame({n: metrics(test_pred.true, test_pred[c]) for n, c in CANDIDATES.items()}).T
display(confirm.astype(float).round(3))
display(pd.DataFrame({n: late_severity(test_pred.true, test_pred[c]) for n, c in CANDIDATES.items()}).T.astype(float).round(2))
better = confirm.loc["Blend RSF + GB", "NASA / engine"] < confirm.loc["RSF (deployed)", "NASA / engine"]
if selected:
    print("Test-set rule -- blend NASA lower than deployed:", "PASSED" if better else "FAILED")
else:
    print("Not selected on cross-validation; test result recorded only. Blend NASA lower than deployed:", better)

,NASA / engine,RMSE,MAE,late rate,C-index
RSF (deployed),22.413,28.206,20.655,0.484,0.862
Gradient boosting (reference),18.865,26.377,18.606,0.468,0.877
Blend RSF + GB,19.401,26.953,19.152,0.476,0.875


,late (any),mean late error,late by >20,late by >30,worst late,near failure (<= 25): late by >10,near failure (<= 25): worst late
RSF (deployed),120.0,13.36,27.0,11.0,50.20,12.0,38.10
Gradient boosting (reference),116.0,11.50,23.0,8.0,42.53,9.0,23.92
Blend RSF + GB,118.0,11.94,21.0,11.0,44.46,8.0,31.01


Not selected on cross-validation; test result recorded only. Blend NASA lower than deployed: True


## 3. Why rules (a) and (b) failed

The cross-validated NASA scores in section 1 are in the hundred-thousands, against ~15 in the range where the model is used. The check below shows where they come from.

In [6]:
def nasa_penalty(y, p):
    d = np.asarray(p, float) - np.asarray(y, float)
    return np.where(d > 0, np.exp(d / 10) - 1, np.exp(-d / 13) - 1)

pen = oof.assign(**{f"pen_{c}": nasa_penalty(oof.true_rul, oof[c]) for c in ("rsf", "blend")})
far = pen.true_rul > 150
print(f"Held-out points with more than 150 cycles left: {far.sum()} of {len(pen)}; longest {pen.true_rul.max():.0f} cycles")
print(f"Share of RSF's total CV NASA penalty from those points: {pen.loc[far, 'pen_rsf'].sum() / pen.pen_rsf.sum():.4%}")
print(f"Mean prediction on those points: RSF {pen.loc[far, 'rsf'].mean():.1f}, blend {pen.loc[far, 'blend'].mean():.1f}"
      f" (true {pen.loc[far, 'true_rul'].mean():.1f}): every model sits just under the 125-cycle cap")
top = pen.nlargest(5, "pen_rsf")[["fold", "unit_number", "true_rul", "rsf", "blend", "pen_rsf", "pen_blend"]]
print(f"Largest single point: {top.pen_rsf.iloc[0] / pen.pen_rsf.sum():.1%} of RSF's total penalty")
display(top.round(1))
near = pen[~far]
per_fold_near = near.groupby("fold").apply(
    lambda g: pd.Series({"RSF": evaluation.nasa_score_mean(g.true_rul, g.rsf),
                         "Blend": evaluation.nasa_score_mean(g.true_rul, g.blend)}), include_groups=False)
display(per_fold_near.round(2))
print(f"Blend better at <= 150 cycles left in {int((per_fold_near.Blend < per_fold_near.RSF).sum())} of 5 folds")
print("Decision: FD004 keeps the RSF point estimate (the pre-set rule was not met); the blend is recorded, not deployed")

Held-out points with more than 150 cycles left: 200 of 1245; longest 366 cycles
Share of RSF's total CV NASA penalty from those points: 99.9911%
Mean prediction on those points: RSF 116.7, blend 117.4 (true 186.6): every model sits just under the 125-cycle cap
Largest single point: 95.1% of RSF's total penalty


,fold,unit_number,true_rul,rsf,blend,pen_rsf,pen_blend
423,1,118,366,119.9,119.5,166715553.4,171420807.5
473,1,118,321,120.5,123.2,4994300.5,4062186.5
326,1,133,308,122.9,124.2,1527832.5,1380048.6
35,0,179,255,79.8,73.0,713091.4,1206634.1
135,0,179,249,77.2,72.6,549640.8,783373.6


,RSF,Blend
fold,,
0,15.12,9.61
1,10.54,6.40
2,15.15,8.50
3,10.48,7.99
4,23.03,15.63


Blend better at <= 150 cycles left in 5 of 5 folds
Decision: FD004 keeps the RSF point estimate (the pre-set rule was not met); the blend is recorded, not deployed


## 4. Decision and findings

**Decision: FD004 keeps the RSF-only point estimate.** The blend failed rules (a) and (b), which were fixed before the run, so it is not deployed, and the retraining gate was not run. The rule is not re-written after the fact on the same data, even though section 3 shows that the failure says little about the model: a rule changed once it has failed, by someone who has already seen the test result, would no longer be a test.

**Findings.**
1. **Overall cross-validated NASA is not measurable on FD004 with this protocol.** FD004 engines live up to ~540 cycles, so scoring at 30–90% of life produces points with up to 366 cycles left. Every model predicts just under the 125-cycle cap there, and the early penalty grows exponentially, so 99.99% of the overall score comes from points with more than 150 cycles left, and a single held-out point (engine #118 with 366 cycles left, RSF 119.9 vs blend 119.5) makes up 95% of it. Rules (a) and (b) were decided by a 0.4-cycle difference on one engine. The same effect exists on FD001 (92% of notebook 18's overall score came from points with more than 150 cycles left, longest 241), which is why notebook 18 added rule (c); there the blend won both overall and at ≤150 cycles, so its decision did not hinge on it. FD004's longer-lived engines (up to 543 cycles vs 362) make the effect decisive.
2. **Where the model is used, the blend was better.** On held-out points with at most 150 cycles left it won all 5 folds (NASA 9.6 vs 14.9 per held-out point, RMSE 20.2 vs 22.2); on the official test set, scored once and for the record only: NASA 19.4 vs 22.4 per engine, RMSE 27.0 vs 28.2, C-index 0.875 vs 0.862, with fewer large late errors near failure (8 vs 12 late by more than 10 cycles).
3. **So FD001 and FD004 now differ for a tested reason**: FD001's blend met the pre-set rule and the gate; FD004's did not meet the rule, for a reason this notebook documents.
4. **What should change in future experiments:** rules (a) and (b) should score only points with at most 150 cycles left (rule (c)'s range), or use a capped NASA penalty, set *before* the next experiment. Re-running this experiment under the corrected rule on the same data and the same, already-seen test set would not be independent evidence; that needs new data (or the next unseen subset), so it is left as stated future work rather than done here.

**Scope.** Gradient boosting alone is again a reference only. The regime normalization is fit on all training engines in every fold (unsupervised, identical for both candidates).